# Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import os

In [2]:
# ===================================================
# Google Drive Mount & Unzip to Colab Local Storage
# ===================================================
import zipfile
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define paths (Extraction will be done in Colab local storage)
ZIP_PATH = "/content/drive/MyDrive/insect_dataset.zip"
EXTRACT_PATH = "/content/"  # <--- This is now the current directory of Colab

print("Unzipping from Google Drive to Colab has started...")

try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Dataset has been successfully extracted to the current Colab directory!")

except zipfile.BadZipFile:
    print("Error: The zip file is corrupt or was not uploaded correctly to Drive.")

Mounted at /content/drive
Unzipping from Google Drive to Colab has started...
Dataset has been successfully extracted to the current Colab directory!



# Dataset Path


In [3]:
DATASET_PATH = r"insect_dataset"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


# Train/Test Split (80/20)


In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("\nClass Labels:")
for i, cls in enumerate(class_names):
    print(f"{i} -> {cls}")

NUM_CLASSES = len(class_names)

Found 1800 files belonging to 3 classes.
Using 1440 files for training.
Found 1800 files belonging to 3 classes.
Using 360 files for validation.

Class Labels:
0 -> Aphids
1 -> Army worm
2 -> Healthy


# Save Class Labels with Numbers

In [5]:

labels_path = "insect_labels.txt"
with open(labels_path, "w") as f:
    for i, cls in enumerate(class_names):
        f.write(f"{i} -> {cls}\n")

print(f"Labels with numbers saved in '{labels_path}'.")

Labels with numbers saved in 'insect_labels.txt'.



# Performance Optimization


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


# Data Augmentation Layer


In [7]:

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])



# MobileNetV2 Base Model


In [8]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step



# Transfer Learning Model


In [9]:

inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation='softmax'
)(x)

model = tf.keras.Model(inputs, outputs)


# Compile


In [10]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


# Callbacks


In [11]:
callbacks = [

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2
    ),

    ModelCheckpoint(
        "best_model.keras",
        save_best_only=True
    )
]


# Initial Training


In [12]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 164ms/step - accuracy: 0.7743 - loss: 0.5311 - val_accuracy: 0.9000 - val_loss: 0.2366 - learning_rate: 0.0010
Epoch 2/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.9375 - loss: 0.1876 - val_accuracy: 0.9444 - val_loss: 0.1666 - learning_rate: 0.0010
Epoch 3/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.9618 - loss: 0.1265 - val_accuracy: 0.9500 - val_loss: 0.1489 - learning_rate: 0.0010
Epoch 4/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.9660 - loss: 0.1182 - val_accuracy: 0.9583 - val_loss: 0.1150 - learning_rate: 0.0010
Epoch 5/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.9653 - loss: 0.1062 - val_accuracy: 0.9667 - val_loss: 0.1091 - learning_rate: 0.0010
Epoch 6/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.9736 - loss: 0.0817 - val_accuracy: 0.9722 - val_loss: 0.0949 - learning_rate: 0.0010
Epoch 7/15
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 81ms/step - accuracy: 0.9806 - loss: 0.0723 - val_a


# Fine Tuning


In [13]:

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 16s 115ms/step - accuracy: 0.9257 - loss: 0.2197 - val_accuracy: 0.9750 - val_loss: 0.0688 - learning_rate: 1.0000e-05
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 83ms/step - accuracy: 0.9576 - loss: 0.1184 - val_accuracy: 0.9778 - val_loss: 0.0651 - learning_rate: 1.0000e-05
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - accuracy: 0.9681 - loss: 0.0942 - val_accuracy: 0.9806 - val_loss: 0.0609 - learning_rate: 2.0000e-06
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.9708 - loss: 0.0940 - val_accuracy: 0.9833 - val_loss: 0.0572 - learning_rate: 2.0000e-06
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.9708 - loss: 0.0875 - val_accuracy: 0.9861 - val_loss: 0.0541 - learning_rate: 2.0000e-06
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.9715 - loss: 0.0846 - val_accuracy: 0.9861 - val_loss: 0.0517 - learning_rate: 2.0000e-06
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 101ms/step - accuracy: 0.9

In [14]:

# =====================================
# Final Evaluation
# =====================================

loss, acc = model.evaluate(test_ds)

print(f"\nFinal Accuracy: {acc*100:.2f}%")

# =====================================
# Save Model
# =====================================

model.save("cotton_insect_mobilenetv2.keras")

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.9889 - loss: 0.0442

Final Accuracy: 98.89%



# TFLite Conversion + Optimization



In [15]:
import tensorflow as tf
import os

# Load Saved Model
model = tf.keras.models.load_model(
    "cotton_insect_mobilenetv2.keras"
)


# Float16 Quantization


In [16]:

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_types = [
    tf.float16
]

tflite_quant_model = converter.convert()

with open(
    "cotton_insect_mobilenetv2_float16.tflite",
    "wb"
) as f:
    f.write(tflite_quant_model)

print("Float16 Quantized Model Saved")

Saved artifact at '/tmp/tmpvkh6enru'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134188244272336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244273488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244272912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244274256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244274832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244275408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244275600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244274448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244275216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134188244276176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1341882442727


# Model Size Comparison


In [17]:


tflite_path="cotton_insect_mobilenetv2_float16.tflite"

keras_size = os.path.getsize(
    "cotton_insect_mobilenetv2.keras"
) / (1024 * 1024)

tflite_size = os.path.getsize(
    tflite_path
) / (1024 * 1024)

print("\nModel Saved Successfully")
print(f"Keras Model Size  : {keras_size:.2f} MB")
print(f"TFLite Model Size : {tflite_size:.2f} MB")


Model Saved Successfully
Keras Model Size  : 20.90 MB
TFLite Model Size : 4.26 MB
